# Agents

Các tác nhân (agent) kết hợp mô hình ngôn ngữ với các công cụ để tạo ra những hệ thống có thể suy luận về nhiệm vụ, quyết định nên sử dụng công cụ nào và làm việc theo từng bước để tiến tới lời giải.

Một LLM Agent vận hành các công cụ theo vòng lặp nhằm đạt được mục tiêu. Tác nhân sẽ tiếp tục chạy cho đến khi đạt điều kiện dừng — ví dụ như khi mô hình tạo ra kết quả cuối cùng hoặc khi chạm đến giới hạn số vòng lặp.

![Agent](./images/agent.png)

# Tools 
Các công cụ (tools) trao cho agent khả năng thực hiện hành động. Agent vượt xa việc chỉ đơn thuần “gắn” mô hình với công cụ bằng cách hỗ trợ:
- Gọi nhiều công cụ theo chuỗi (được kích hoạt từ một prompt duy nhất)
- Gọi công cụ song song khi phù hợp
- Lựa chọn công cụ một cách động dựa trên các kết quả trước đó
- Cơ chế thử lại công cụ và xử lý lỗi

Duy trì trạng thái xuyên suốt các lần gọi công cụ

## Defining tools
Pass a list of tools to the agent.

In [1]:
from langchain.tools import tool
from langchain.agents import create_agent

### Search Tool 
https://docs.langchain.com/oss/python/integrations/tools


In [2]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Rikkeisoft?")

"February 5, 2026 - Rikkeisoft (Công ty Cổ phần Rikkeisoft) là công ty công nghệ thông tin tại Việt Nam được thành lập vào năm 2012 với lĩnh vực kinh doanh chính là cung cấp các dịch vụ và giải pháp công nghệ thông tin . Hiện tại, Rikkeisoft có 05 văn phòng làm ... January 13, 2026 - Rikkeisoft – A leading Vietnamese IT company providing software development, digital transformation, and AI solutions worldwide . December 22, 2025 - From 2024, I am CEO of Rikkeisoft. October 16, 2025 - We appreciate your interest in Rikkeisoft. Send us a question, and we'll get back to you as soon as possible. May 29, 2025 - Rikkeisoft – A leading Vietnamese IT company providing software development, digital transformation, and AI solutions worldwide ."

In [3]:
from langchain_community.tools import DuckDuckGoSearchResults
search = DuckDuckGoSearchResults(output_format="list")

search.invoke("Rikkeisoft?")

[{'snippet': 'In 2012, Rikkeisoft was established by six software developers on a lofty mission: to drive technological innovation and deliver lasting values. Following in the footsteps of our founders, we cherish the tradition of excellence.',
  'title': 'Rikkeisoft - Trusted IT Outsourcing Provider',
  'link': 'https://rikkeisoft.com/'},
 {'snippet': 'Rikkeisoft tự hào thông báo: Rikkei Japan - pháp nhân của Rikkeisoft tại Nhật Bản đã chính thức được công nhận là “Great Place to Work 2025”, danh hiệu uy tín toàn cầu dành cho các tổ chức có môi trường làm việc xuất sắc! Lần đầu tiên có mặt trong danh sách này, Rikkei Japan Established in 2012, Rikkeisoft is a leading provider of technology resources & services for the US, Europe, and Asia-Pacific (APAC) region. For 10 years+, we have been helping businesses &... Founded in 2012, Rikkeisoft is the largest private technology company in Vietnam, specializing in helping customers with digitalization and innovative solutions across the U.S

In [4]:
@tool(description="Search for information about Rikkeisoft.")
def search_rikkeisoft_information(query: str) -> str:
    """Search for information about Rikkeisoft."""
    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    result = search.invoke(query)
    if result:
        return result
    else:
        return "No information found"

search_rikkeisoft_information.invoke({
    "query": "Rikkeisoft?"
})





'250+ reviews môi trường làm việc, văn hoá, mức lương tại RIKKEISOFT . Được đăng ẩn danh bởi nhân viên làm việc tại đây RikkeiSoft - TPHCM, Ho Chi Minh City. 1,494 likes · 5 talking about this · 125 were here. Where the dream begins Rikkeisoft is an unfunded company based in Hanoi (Vietnam), founded in 2012. It operates as a Provider of web & mobile application development, cloud, AI, testing, and IT managed services. Ông Tạ Sơn Tùng, Chủ tịch Rikkeisoft chia sẻ, Rikkeisoft đang chuẩn bị cho cột mốc IPO tại Nhật trong 3 năm tới và hướng tới giấc mơ trở thành kỳ lân công nghệ Việt Nam. Rikkeisoft , nhà cung cấp dịch vụ phần mềm thuê ngoài (outsourcing), chuyển hướng sang cung cấp giải pháp, đầu tư vào AI với mục tiêu trở thành kỳ lân công nghệ.'

### Get current time tool

In [5]:
@tool(description="Get current time by UTC offset.")
def get_current_time(utc: int = 0) -> str:
    """Get current time by UTC offset."""
    from datetime import datetime, timedelta, timezone
    tz = timezone(timedelta(hours=utc))
    return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")


In [6]:

@tool
def calculate_years_of_establishment(start_year: int) -> str:
    """Calculate the number of years since the establishment of Rikkeisoft."""
    from datetime import datetime
    return f"Rikkeisoft was established in {datetime.now().year - start_year} years ago"

calculate_years_of_establishment.invoke({
    "start_year": 2010
})

'Rikkeisoft was established in 16 years ago'

## Model
The model is the reasoning engine of your agent. It can be specified in multiple ways, supporting both static and dynamic model selection.

### Static model
Static models are configured once when creating the agent and remain unchanged throughout execution. This is the most common and straightforward approach.

In [12]:
from config import settings
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model=settings.LLM_CHAT_MODEL,api_key=settings.LLM_API_KEY,base_url=settings.LLM_BASE_URL)

agent = create_agent(model=model, tools=[
    search_rikkeisoft_information,
    get_current_time,
    calculate_years_of_establishment
])


In [13]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current time in Vietnam?"}]}
)

In [14]:
result

{'messages': [HumanMessage(content='what is the current time in Vietnam?', additional_kwargs={}, response_metadata={}, id='d380eab0-3097-4d0e-96a5-0f48dbe067eb'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"utc": 7}'}, '__gemini_function_call_thought_signatures__': {'3fe77ef5-9301-4693-a3f9-c35c9a203385': 'EpADCo0DAb4+9vvz93r1c80rZ50ISMiCvp1/1/u49YUTWeiFeSYL9Rv95X77jfw/Go1mhAGweHIglAOCJl2xvJm22o+t6QOv2ElRKFZBy4JkSwMQG46mAMgBpA6h7U7Zv2gxI93ftueXfKCpR4uBhTsxv1jm5nXYr+w0a6SNnRg57E4IT2vJyVxp25Yts/kmhn/nBTcN6xnA2klSekXtxOhMD+6TzRA5kgdJGhd3xnRoM1tzgqEusQKI60mI62ErFd66wrwq2twzA5w0W+rl85f5oiB1uUZRJGqqRtDWjZfmZFJ1LvNRqAphjvZWRqni/bQOZQ89tRG5mNMryiseJwaSan68oKijW7/NwYA3gtmhaqs59jobJ8/VfOwCZyS84UpqWulAIX/ho9W+GxzkhoXxlVg2GBAzljHEJX50A0JLX2YoanIrQgkcwbob/yym31jAgPNwqG1Y+DtoUZ1azxNwA9XGYLFl4C3zLF2C2weUd4iZK667y468AqOtsHXEWU3qc0A8kOXDu5DcbrXMGG7FUw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 's

In [15]:
final_message = result["messages"][-1].content
print(final_message)

[{'type': 'text', 'text': 'The current time in Vietnam is 22:45:02 (10:45 PM) on March 24, 2026.'}]


In [16]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]}
)
result
final_message = result["messages"][-1].content
print(final_message)


[{'type': 'text', 'text': 'Rikkeisoft được thành lập vào ngày 06/04/2012. Tính đến thời điểm hiện tại (tháng 03/2026), Rikkeisoft đã hoạt động được gần **14 năm**.', 'extras': {'signature': 'Er8DCrwDAb4+9vuPII/EdqpRlfBm0r+YIXkb46KZ8y5RVXSa1wPjQ28Z4TBsivkuz/Ys6X+GinARsR0gbVx3CvnOmDO0ewvW77ct2aWGk4PSfmEupCSGnoIzuT4aDq1xvC4UshzsEgb6UV3nEhwnFVljqVu6RHKrzpO2MfNUzCe50RlDCFgGbiZosYk2MCQih5qkCwnfPt99UtQbzFFIX0l/aGoSuszg+Q1aXydot+uliCMlPXd+6AmBI+Cq1Il8tGByYQ+lBh64KXsGnXSqhIOzIFEETdv9kUDo9sLY+b6758fpic+BzupbKH4V5BjoHquV0nozgFJxLJNcG347+zIi8VZ4gyrDDcHnfpBPdIPg8xCZuw76rZzNFfc+pA70nootngf8nu8kHZKXe47Zu4bml+WapFAxSHUXA46MH0DWnIxIGH3PAep5oRjcorode/6cSSd3znUXlE6K26xj/+VTwYJLvAr1WgfnFKwNvAtSwd+VUI2i4Ss/tTVX/PZlW80gFbnamXtqQEe0GTTwe/cmcnX+ppifps1WPlOvZqFW4PE7dEShnZIUkNohaKZ2rzIpEROZWA3BKXzVVe3fhmRb'}}]


In [17]:
result

{'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='d0b27af3-331d-45a8-8bbe-2eb31761852b'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "n\\u0103m th\\u00e0nh l\\u1eadp Rikkeisoft"}'}, '__gemini_function_call_thought_signatures__': {'24d74e06-0bb9-46da-b85c-aa3d6d262076': 'EuYBCuMBAb4+9vv57aVAZPxYFZd3IYVNL5BbrJ/52PVr8+7lbG3KyImL//Mtx+/XkAK3YRHqekY3nY98uPkY96LvcXn8f4zqApZA1zpQNy/1USjjb3n/oAeBQ/Q9GBJNtbNZcxxkk4BCt7X/v+mDiPI86VUhDUdubtsHeapEhjTQMJKsHLUCgC3k2OjFR/C7IHGQyZOaUvEFsMRkuoccqWe81Di7Jd/XzdqK+O0pTvgI7fvWZSRxwm4cI6vdxv1G2iB31/VArgGy085OVLOdezjlMrncyIZ4lRRAW70NQm4KOB/H0Xf+sFU='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d2086-9d61-7371-8907-e211f421550d-0', tool_calls=[{'name': 'search_rikkeisoft_information', 

### Dynamic Model là gì?

**Dynamic model** là cơ chế cho phép **chọn mô hình LLM tại thời điểm runtime** dựa trên **ngữ cảnh, trạng thái hiện tại hoặc logic tùy biến** (ví dụ: độ phức tạp câu hỏi, chi phí, độ trễ, người dùng trả phí hay miễn phí).  
Thay vì cố định một model (như GPT-4 hoặc Gemini-Pro), hệ thống có thể **tự động chuyển đổi model** để đạt hiệu quả tốt nhất giữa **chất lượng – chi phí – tốc độ**.

---

### Vì sao cần Dynamic Model?

Dynamic model giúp:
- **Routing thông minh**: câu hỏi đơn giản dùng model rẻ/nhanh, câu hỏi phức tạp dùng model mạnh.
-  **Tối ưu chi phí**: giảm dùng model đắt khi không cần thiết.
-  **Cải thiện hiệu năng**: ưu tiên model phản hồi nhanh trong các tình huống realtime.
- **Linh hoạt mở rộng**: dễ thêm model mới mà không thay đổi toàn bộ hệ thống.

### Middleware với `@wrap_model_call`


Để dùng dynamic model, bạn tạo middleware bằng decorator `@wrap_model_call`. Middleware này sẽ **chỉnh sửa model trong request** trước khi LLM được gọi.


In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatGoogleGenerativeAI(api_key=settings.LLM_API_KEY,model="gemini-1-gemini-2.5-flash", base_url=settings.LLM_BASE_URL)
advanced_model = ChatGoogleGenerativeAI(api_key=settings.LLM_API_KEY,model="gemini-1-gemini-2.5-pro", base_url=settings.LLM_BASE_URL)

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    middleware=[dynamic_model_selection]
)

In [19]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]}
)
result
final_message = result["messages"][-1].content
print(final_message)

[{'type': 'text', 'text': 'Rikkeisoft được thành lập cách đây 14 năm.', 'extras': {'signature': 'CpcCAb4+9vsA8TalZkhdMOQIM9aMQG5bkRjau+FtUCBWpYTd8Zg7RSqS38APHAao3r370+B5dMS9IQspiSJB6s1qyjevHEbmb5CfN6jL+Luiy5A4E40Ws+sGXLou3Ov6Bl9PzvHWVMXtyLfaDzRWEvEg5ZX02m1a0NXDULpOUXcyzo9gv14rX8f7xT4yNWAzCWHKRaoqwEFzdP90k+bE+gv0KyxuzJigVvnlXTn7nJLKTKjhx4yDs5IXiXMH5wuHPf9JLdRVl9ikAhcABVARaKoX5U81A5gk8A8ikW35yaHN96+FvSF4Ojz6DACHqDMFTjVAO47LLH0aJJpOlofYRbd4+sVCBZ+1QGxJ/4rgRw+0wQEabuQHbOTM'}}]


### System Prompt 
**System prompt** là thông điệp dùng để **định hình hành vi, vai trò và phong cách làm việc của agent** ngay từ đầu.  
Nó trả lời cho câu hỏi: *“Agent này nên suy nghĩ và phản hồi như thế nào?”*
Trong LangChain, có thể truyền system prompt khi tạo agent để kiểm soát:
- Cách agent tiếp cận nhiệm vụ
- Mức độ chi tiết / ngắn gọn
- Tính cách, vai trò (assistant, chuyên gia, reviewer, v.v.)

In [20]:
agent = create_agent(
    model=model,  # Default model
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    middleware=[dynamic_model_selection],
    system_prompt="You are a helpful assistant that can answer questions and help with tasks."
)

In [21]:
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current time?"}]}
)

{'messages': [HumanMessage(content='what is the current time?', additional_kwargs={}, response_metadata={}, id='9d2c32e8-3daf-487b-98a0-7ca02f64144b'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'6d7950fb-0f9e-4ce0-babf-29da9da183fc': 'Ct8BAb4+9vvbwWbdXeasRTH7/kWSWX2jsWXoYjmDZE2k3fepeHRBrBvmjCHBzotoZewQpXPq597AEwJ9eFoTEY0ob0OuXhC60ELq7Z5uICBxUK5yG1SoZBB/TmqNCQtLp4fxz/pmnJEZyF78mGh+1f0xn35AOzqss/2ixfxrd9a3VPVj+ypH8fb4ITp93Tc+Ls1lR3AWZyDRxnUkplIK2eCJDpUDMxjtaCx5CWZNsZCSscHAMkuv+/1fGBkWqdvlKgYRATjTbqFwnTjhpnBkTAw4ojznCh4LRl7ZgY/mq39L5A=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d2088-877c-7520-a607-f513d77bab6c-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': '6d7950fb-0f9e-4ce0-babf-29da9da183fc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me

### Memory

Trong LangChain, **Agent tự động duy trì lịch sử hội thoại** thông qua *message state*.  
Phần thông tin này có thể xem như **short-term memory** (bộ nhớ ngắn hạn) của agent, giúp agent:
- Hiểu ngữ cảnh cuộc trò chuyện
- Trả lời nhất quán qua nhiều lượt
- Tham chiếu lại thông tin đã nói trước đó


In [25]:
from langgraph.checkpoint.memory import MemorySaver  
checkpointer = MemorySaver()  # In-memory 
from typing import Any
agent = create_agent(
    model,
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    checkpointer=checkpointer
)
config = {"configurable": {"thread_id": "session_1"}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}],
}, config)

In [26]:
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'Rikkeisoft được thành lập vào năm **2012**. Tính đến năm 2026, Rikkeisoft đã có **14 năm** hình thành và phát triển.', 'extras': {'signature': 'Es8BCswBAb4+9vvi1eclsvYdisAf+rczu1MK35BqMWLCVEOcQvU/V2db5S9LNhKa4+bA0LDA0fQsvb42q58JzKa8wmFhowkBJEwQOFDWPHsibvZdqCno8/FejHFv6kzlIYCz32TFSdTMnkorl551n9+H6QIg/SBwcQFRdLHqtWmsAp6iMF2RrZgUeluVEbuRTK8nBlGRg36jAyj2TORUMmkvi/DxKZdY8fSxCyL97cU8I1Rxkrt/vB6t75QLnqt04mQtwJMWv/h505rY9uS903fG'}}]


In [27]:
#print checkpointer
checkpointer.get(config)

{'v': 4,
 'ts': '2026-03-24T15:50:18.422591+00:00',
 'id': '1f127992-8441-647e-8007-47a8c7180b23',
 'channel_versions': {'__start__': '00000000000000000000000000000002.0.25347472755411415',
  'messages': '00000000000000000000000000000009.0.6499747896018543',
  'branch:to:model': '00000000000000000000000000000009.0.6499747896018543',
  '__pregel_tasks': '00000000000000000000000000000008.0.4719708250652065'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000001.0.9965913552319414'},
  'model': {'branch:to:model': '00000000000000000000000000000008.0.4719708250652065'},
  'tools': {}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='06daca9d-7942-416d-a818-bb46314db35d'),
   AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "n\\u0103m th

In [28]:
result_2 = agent.invoke({
    "messages": [{"role": "user", "content": "Tôi đã hỏi bạn gì nhỉ?"}],
}, config)
print(result_2["messages"][-1].content)
checkpointer.get(config)


[{'type': 'text', 'text': 'Bạn vừa hỏi tôi là: **"Rikkeisoft được thành lập bao nhiêu năm rồi?"**\n\nTôi đã trả lời rằng Rikkeisoft được thành lập vào năm 2012, và tính đến năm 2026 thì công ty đã hoạt động được 14 năm.', 'extras': {'signature': 'EusCCugCAb4+9vum2SLCNPqKWHNS4wArlY40/vjWucoJ8Wt3NrrnhG5B1EHQz+IDwYlBfZGhaPToK+lR6iPLNizJ6iw0Jg8gIP94hXddIvWyIGHAgNK03Rt0Fqq1oXbgxsbICrfFw9WQeZmS5J5mzBiiYFirK9thHuyIx75matcYeGQ+RIKbmAD2VfsJAqLPGJ/rtAQUyf4E89cNJD0yaXkd3cGtdq6FqG55ACmDnX0cfqBAWFs0yeBE0GnTCgBeUFfiSInT9+7b1peppFaU2FiZT0xyciCT4unDUmOrLagRN0G+QztSYDVRdxztLP1XX19ZqZcCe9KVmYNupfv2yaqcomAlMXHgrbpsJRGqVys9xGnQBV77xyvukrYsoyyoxOUdX0P2PXx49wdUmuQ2lWcpmPk2vuBJrrJ2FJ35s/wXSBEYkUlFXpTkX7PfU2F7agCrc/A7RWV13QvNtXcibsRWGeRaeOVXpAt+mqgv'}}]


{'v': 4,
 'ts': '2026-03-24T15:50:20.332885+00:00',
 'id': '1f127992-9679-6155-800a-ff8cfc420aaa',
 'channel_versions': {'__start__': '00000000000000000000000000000011.0.32343245238201324',
  'messages': '00000000000000000000000000000012.0.46535829760770997',
  'branch:to:model': '00000000000000000000000000000012.0.46535829760770997',
  '__pregel_tasks': '00000000000000000000000000000008.0.4719708250652065'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000010.0.28317838124212624'},
  'model': {'branch:to:model': '00000000000000000000000000000011.0.32343245238201324'},
  'tools': {}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='06daca9d-7942-416d-a818-bb46314db35d'),
   AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "n\\u0103

In [29]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Bây giờ là mấy giờ Nhật?"}]
}, config, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

Agent: Bây giờ là mấy giờ Nhật?
Calling tools: ['get_current_time']
Agent: 2026-03-25 00:50:24
Agent: [{'type': 'text', 'text': 'Bây giờ tại Nhật Bản là **00:50**, thứ Tư ngày 25 tháng 03 năm 2026.'}]
